<h1 style=\"text-align: center; font-size: 50px;\">🎥 Advanced Recommender Systems with Tensorflow MLflow Integration</h1>

## Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Loading Data
- Memory-Based Collaborative Filtering
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2025-10-09 16:25:18 - INFO - Notebook execution started.


## User Constants

In [3]:
MOVIE_ID = 5
RATING = 3.5

## Install and Import Libraries

In [4]:
%%time

%pip install -r ../requirements.txt --quiet

# ------------------------ Data Manipulation ------------------------
import numpy as np
import pandas as pd

# # ------------------------ Statistical and Machine Learning tools ------------------------
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import pairwise_distances
from sklearn.metrics import mean_squared_error
from math import sqrt
import scipy.sparse as sp
from scipy.sparse.linalg import svds

# ------------------------ Deep learning framework ------------------------
import tensorflow as tf
from tensorflow.keras.callbacks import TensorBoard

# ------------------------ System Utilities ------------------------
import os
import warnings
import datetime
from pathlib import Path
import sys

# ------------------------ Visualization Libraries ------------------------
import matplotlib.pyplot as plt

# ------------------------ MLflow Integration ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.types.schema import Schema, ColSpec
from mlflow.models import ModelSignature

# ------------------------ Utils Import ------------------------
sys.path.append("../src")
from utils import load_config

Note: you may need to restart the kernel to use updated packages.


2025-10-09 16:25:23.071046: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-10-09 16:25:23.330019: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760027123.419473     470 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760027123.439970     470 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-10-09 16:25:23.661668: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

CPU times: user 6.18 s, sys: 4.36 s, total: 10.5 s
Wall time: 12.3 s


## Configure Settings

In [5]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [6]:
# ------------------------- Paths -------------------------
DATA_PATH = "/home/jovyan/datafabric/tutorial/"
LOG_DIR = "/phoenix/tensorboard/tensorlogs/"
OUTPUT_DIR = "../model_artifacts"
ARTIFACT_PATH =  "movie_titles" 
# Name of the MLflow experiment for tracking performance and metrics
EXPERIMENT_NAME = "MovieRecommenderExperiment"
RUN_NAME = "Movie_Recommender_Run"
MODEL_NAME = "movie_titles"     

# Configuration paths
CONFIG_PATH = "../configs/config.yaml"
DEMO_FOLDER = "../demo"

# Load configuration
config = load_config(CONFIG_PATH)

logger.info("✅ Configuration loaded successfully")

2025-10-09 16:25:30 - INFO - ✅ Configuration loaded successfully


## Verify Assets

In [7]:
# Check whether the Dataset file exists
is_dataset_available = Path(DATA_PATH).exists()

# Log the configuration status of the dataset
if is_dataset_available:
    logger.info("The Dataset is properly configured.")
else:
    logger.info(
        "The Dataset is not properly configured. Please create and download the required assets "
        "in your project on AI Studio."
    )

2025-10-09 16:25:30 - INFO - The Dataset is properly configured.


## Loading Data

In [8]:
asset_folder = DATA_PATH

In [9]:
column_names = ['user_id', 'item_id', 'rating', 'timestamp']
df = pd.read_csv(f"{asset_folder}ml-100k/u.data", sep='\t', names=column_names)

In [10]:
movie_titles = pd.read_csv(f"{asset_folder}Movie_Id_Titles.csv")

In [11]:
df = pd.merge(df,movie_titles,on='item_id')

In [12]:
display(df.sample(2))
display(df.shape)

,user_id,item_id,rating,timestamp,title
97220,571,144,2,883354992,Die Hard (1988)
65750,751,490,4,889133429,To Catch a Thief (1955)


(100000, 5)

In [13]:
train_data, test_data = train_test_split(df, test_size=0.25)

## Memory-Based Collaborative Filtering

In [14]:
n_users = df.user_id.nunique()
n_items = df.item_id.nunique()
#Create two user-item matrices, one for training and another for testing
train_data_matrix = np.zeros((n_users, n_items))
for line in train_data.itertuples():
    train_data_matrix[line[1]-1, line[2]-1] = line[3]  

test_data_matrix = np.zeros((n_users, n_items))
for line in test_data.itertuples():
    test_data_matrix[line[1]-1, line[2]-1] = line[3]

In [15]:
user_similarity = pairwise_distances(train_data_matrix, metric='cosine')
item_similarity = pairwise_distances(train_data_matrix.T, metric='cosine')

In [16]:
def predict(ratings, similarity, type='user'):
    """
    Predicts ratings using collaborative filtering based on user or item similarity.

    Parameters:
        ratings (array): A matrix where each row represents a user and each column represents an item.
        similarity (array): A similarity matrix representing relationships between users or items.
        type (str): Defines the type of prediction. Defaults to 'user'.

    Returns:
        array: A matrix of predicted ratings.
    """
    try:
        if type == 'user':
            mean_user_rating = ratings.mean(axis=1)
            #You use np.newaxis so that mean_user_rating has same format as ratings
            ratings_diff = (ratings - mean_user_rating[:, np.newaxis]) 
            pred = mean_user_rating[:, np.newaxis] + similarity.dot(ratings_diff) / np.array([np.abs(similarity).sum(axis=1)]).T
        elif type == 'item':
            pred = ratings.dot(similarity) / np.array([np.abs(similarity).sum(axis=1)])     
        return pred
    except Exception as e:
        logger.error(f"Error predicting ratings: {str(e)}")
        raise

In [17]:
item_prediction = predict(train_data_matrix, item_similarity, type='item')
user_prediction = predict(train_data_matrix, user_similarity, type='user')

### SVD

In [18]:
def rmse(prediction, ground_truth):
    """
    Computes the Root Mean Square Error (RMSE) between predicted values and ground truth values.

    Parameters:
        prediction (array-like): Predicted values.
        ground_truth (array-like): Actual values.

    Returns:
        float: The RMSE value.
    """
    try:
        prediction = prediction[ground_truth.nonzero()].flatten() 
        ground_truth = ground_truth[ground_truth.nonzero()].flatten()
        return sqrt(mean_squared_error(prediction, ground_truth))
    except Exception as e:
            logger.error(f"Error computing rmse: {str(e)}")
            raise

In [19]:
#get SVD components from train matrix. Choose k.
u, s, vt = svds(train_data_matrix, k = 20)
s_diag_matrix=np.diag(s)
X_pred = np.dot(np.dot(u, s_diag_matrix), vt)
logger.info('User-based CF MSE: ' + str(rmse(X_pred, test_data_matrix)))

2025-10-09 16:25:32 - INFO - User-based CF MSE: 2.7257971163559405


## Logging Model to MLflow

In [20]:
def normalize_ratings(ratings, min_rating=1, max_rating=5):
    """Normalize ratings to 1-5 scale"""
    ratings = np.array(ratings)
    
    if ratings.max() == ratings.min():
        return np.full(ratings.shape, 3.0)  # Return middle value
    
    normalized = (ratings - ratings.min()) / (ratings.max() - ratings.min())
    scaled = normalized * (max_rating - min_rating) + min_rating
    
    return np.clip(scaled, min_rating, max_rating)

class MovieRecommender(mlflow.pyfunc.PythonModel):
    def get_movie_title(self, movie_id):
        """
        Returns the movie title for a given movie_id.

        Parameters:
            movie_id (int): The ID of the movie.

        Returns:
            str: The title of the movie, or None if not found.
        """
        try:
            title_row = self.movie_titles[self.movie_titles['item_id'] == movie_id]
            if not title_row.empty:
                return title_row.iloc[0]['title']
            else:
                logger.warning(f"⚠️ Movie ID {movie_id} not found in titles.")
                return None
        except Exception as e:
            logger.error(f"❌ Error retrieving movie title: {str(e)}")
            return None

    def load_context(self, context):
        """
        Load model and configuration from artifacts.
        """
        try:
            import yaml
            
            self.train_data_matrix = np.load(context.artifacts["train_data_matrix"])
            self.movie_titles = pd.read_csv(context.artifacts["movie_titles_path"])
            
            # Load settings from config file
            with open(context.artifacts["config"], 'r') as f:
                config = yaml.safe_load(f)
            
            # Get settings (or use defaults if not in config)
            self.neutral_rating =  3.0
            self.personalization_weight =  2.0
            self.top_n =  5
            
            # Replace zeros with NaN so they don't affect the average
            data_with_nan = np.where(self.train_data_matrix > 0, 
                                      self.train_data_matrix, 
                                      np.nan)
            self.mean_ratings = np.nanmean(data_with_nan, axis=0)
            self.mean_ratings = np.nan_to_num(self.mean_ratings, nan=0.0)
            
            logger.info("✅ Model and configuration loaded successfully")
        except Exception as e:
            logger.error(f"❌ Error loading context: {str(e)}")
            raise

    def predict(self, context, model_input):
        """
        Performs prediction using specified prediction method.

        Parameters:
            context: MLflow context.
            model_input: Input data

        Returns:
            array: Model prediction output.
        """
        try:
            movie_ids = model_input['movie_id'].tolist()
            ratings = model_input['rating'].tolist()
            
            # Calculate predictions for all movies
            predictions = np.zeros(len(self.mean_ratings))
            
            for movie_id, rating in zip(movie_ids, ratings):
                movie_idx = int(movie_id) - 1  # Convert to array index
                
                # Skip invalid movie IDs
                if movie_idx < 0 or movie_idx >= len(predictions):
                    continue
                
                # How much user likes/dislikes compared to neutral (3.0)
                weight = (float(rating) - self.neutral_rating) / 2.0

                item_similarity = 1 - pairwise_distances(self.train_data_matrix.T, metric='cosine')
                # Add similar movies weighted by user preference
                predictions += item_similarity[movie_idx] * weight
            
            # Combine user preference with popular movies
            final_predictions = self.mean_ratings + predictions * self.personalization_weight
            
            # Normalize to 1-5 scale
            final_predictions = normalize_ratings(final_predictions)
            
            # Create list of (title, score, index)
            movie_titles = self.movie_titles['title'].tolist()
            rated_indices = set(int(mid) - 1 for mid in movie_ids)
            
            all_movies = []
            for idx, (title, score) in enumerate(zip(movie_titles, final_predictions)):
                if idx not in rated_indices:  # Skip already rated
                    all_movies.append((title, float(score)))
            
            # Sort by score and return top N
            all_movies.sort(key=lambda x: x[1], reverse=True)
            return all_movies[:self.top_n]
        
        except Exception as e:
            logger.error(f"❌ Error performing prediction: {str(e)}")
            raise

    @classmethod
    def log_model(cls, train_data_matrix_path, movie_titles_path, config_path, demo_folder):
        """
        Logs the model to MLflow with appropriate artifacts and schema (vanilla-rag pattern).

        Parameters:
            train_data_matrix_path (array): Path to training data.
            movie_titles_path (array): Path to movie titles.
            config_path (str): Path to configuration file.
            demo_folder (str): Path to demo folder.
        """
        try:
            input_schema = Schema([
                ColSpec("long", "movie_id"),
                ColSpec("double", "rating")
            ])
            output_schema = Schema([
                ColSpec("string", "movie_title"),
                ColSpec("double", "prediction")
            ])
            signature = ModelSignature(inputs=input_schema, outputs=output_schema)

            # Prepare artifacts exactly like vanilla-rag does
            artifacts = {
                "train_data_matrix": train_data_matrix_path,
                "movie_titles_path": movie_titles_path,
                "config": config_path
            }
            
            # Add demo folder if it exists (like vanilla-rag)
            if demo_folder and os.path.exists(demo_folder):
                artifacts["demo"] = demo_folder
                logger.info(f"✅ Demo folder added to artifacts: {demo_folder}")

            mlflow.pyfunc.log_model(
                artifact_path=ARTIFACT_PATH,
                python_model=cls(),
                artifacts=artifacts,
                signature=signature,
                pip_requirements=[
                    "mlflow", 
                    "pandas", 
                    "scikit-learn", 
                    "numpy",
                    "streamlit>=1.28.0",
                    "pyyaml"
                ]
            )
            
            logger.info("✅ Model and artifacts successfully registered in MLflow")
        except Exception as e:
            logger.error(f"❌ Error logging model: {str(e)}")
            raise

output_dir = OUTPUT_DIR
os.makedirs(output_dir, exist_ok=True)
train_data_matrix_path = os.path.join(output_dir, "train_data_matrix.npy")
np.save(train_data_matrix_path, train_data_matrix)
movie_titles_path = os.path.join(output_dir, "movie_titles.csv")
movie_titles.to_csv(movie_titles_path, index=False)

/opt/conda/lib/python3.12/site-packages/mlflow/pyfunc/utils/data_validation.py:168: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [21]:
logger.info(f'🚀 Starting the experiment: {EXPERIMENT_NAME}')

# Set the MLflow experiment name
mlflow.set_tracking_uri("/phoenix/mlflow")
mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

# Start an MLflow run
with mlflow.start_run(run_name=RUN_NAME) as run:
    user_rmse = rmse(user_prediction, test_data_matrix)
    item_rmse = rmse(item_prediction, test_data_matrix)
    svd_rmse = rmse(X_pred, test_data_matrix)
    
    mlflow.log_metric("User_based_CF_RMSE", user_rmse)
    mlflow.log_metric("Item_based_CF_RMSE", item_rmse)
    mlflow.log_metric("User_based_CF_MSE_SVD", svd_rmse)
    # Print the artifact URI for reference
    logger.info(f"📁 Run's Artifact URI: {run.info.artifact_uri}")

    # Log the model to MLflow
    MovieRecommender.log_model(
        train_data_matrix_path, 
        movie_titles_path, 
        config_path=CONFIG_PATH,
        demo_folder=DEMO_FOLDER
    )

    # Register the logged model in MLflow Model Registry
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )

    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

logger.info(f'✅ Registered the model: {MODEL_NAME}')

2025-10-09 16:25:32 - INFO - 🚀 Starting the experiment: MovieRecommenderExperiment
2025-10-09 16:25:33 - INFO - 📁 Run's Artifact URI: /phoenix/mlflow/344822691853020071/fb1ecb00c113483da0f98260300b312b/artifacts
2025-10-09 16:25:33 - INFO - ✅ Demo folder added to artifacts: ../demo


2025/10/09 16:25:35 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - streamlit (current: uninstalled, required: streamlit>=1.28.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025-10-09 16:25:35 - INFO - ✅ Model and artifacts successfully registered in MLflow
Registered model 'movie_titles' already exists. Creating a new version of this model...
Created version '2' of model 'movie_titles'.
2025-10-09 16:25:36 - INFO - ✅ Model registered successfully with run ID: fb1ecb00c113483da0f98260300b312b
2025-10-09 16:25:36 - INFO - ✅ Registered the model: movie_titles


## Fetching the Latest Model Version from MLflow

In [22]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the model
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
logger.info(f"Latest Model Version: {latest_model_version}")
logger.info(f"Model Signature: {model_info.signature}")

2025-10-09 16:25:36 - INFO - Latest Model Version: 2
2025-10-09 16:25:36 - INFO - Model Signature: inputs: 
  ['movie_id': long (required), 'rating': double (required)]
outputs: 
  ['movie_title': string (required), 'prediction': double (required)]
params: 
  None



## Loading the Model and Running Inference

In [23]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")

df_input = pd.DataFrame({
   
    'movie_id': [310, 325, 340, 355, 370, 385, 400, 415, 430, 445],
    'rating': [3.0, 2.5, 4.0, 5.0, 1.0, 3.5, 4.5, 2.0, 5.0, 3.0],


})
prediction = model.predict(df_input)
logger.info(prediction)

2025/10/09 16:25:36 WARNING mlflow.utils.requirements_utils: Detected one or more mismatches between the model's dependencies and the current Python environment:
 - streamlit (current: uninstalled, required: streamlit>=1.28.0)
To fix the mismatches, call `mlflow.pyfunc.get_model_dependencies(model_uri)` to fetch the model's environment and install dependencies using the resulting environment file.
2025-10-09 16:25:36 - INFO - ✅ Model and configuration loaded successfully
2025-10-09 16:25:37 - INFO - [('Great Day in Harlem, A (1994)', 4.920917739877747), ('Star Kid (1997)', 4.459362751677223), ('Rear Window (1954)', 4.420158397084226), ('Godfather: Part II, The (1974)', 4.419433898957626), ('Aiqing wansui (1994)', 4.396422393774929)]


In [24]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-10-09 16:25:37 - INFO - ⏱️ Total execution time: 0m 19.00s
2025-10-09 16:25:37 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**Z by HP AI Studio**](https://zdocs.datascience.hp.com/docs/aistudio/overview).